In [29]:
import requests
import pandas as pd
import time

In [30]:
roles = [
    "Data Analyst",
    "Data Scientist",
    "Data Engineer",
    "Cloud Engineer",
    "Software Engineer"
]

all_jobs = []

for role in roles:

    print(f"\nCollecting {role}")

    cursor = None

    # 4 batches × 5 pages = up to 20 pages
    for batch in range(4):

        params = {
            "query": f"{role} jobs in Philippines",
            "num_pages": 5,
            "country": "ph"
        }

        # Continue from previous batch
        if cursor:
            params["cursor"] = cursor

        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=60
        )

        print(
            f"Batch {batch + 1} | "
            f"HTTP {response.status_code}"
        )

        # If request failed, skip this batch
        if response.status_code != 200:
            print("Request failed:", response.text)
            continue

        data = response.json()

        # Make sure expected data exists
        if "data" not in data:
            print("No data returned")
            print(data)
            continue

        jobs_data = data["data"]

        jobs = jobs_data.get("jobs", [])

        print(f"Jobs returned: {len(jobs)}")

        # Stop if API returns nothing
        if len(jobs) == 0:
            break

        df = pd.DataFrame(jobs)

        df["search_role"] = role

        all_jobs.append(df)

        # Get cursor for next batch
        cursor = jobs_data.get("cursor")

        # If there is no next cursor, stop
        if not cursor:
            print("No more pages.")
            break

        # Small pause between API calls
        time.sleep(1)


# Combine everything
combined_df = pd.concat(
    all_jobs,
    ignore_index=True
)

print("\nRaw shape:")
print(combined_df.shape)


# Remove exact duplicate postings
combined_df = combined_df.drop_duplicates(
    subset="job_id"
)

print("\nAfter duplicates:")
print(combined_df.shape)


# Save
combined_df.to_csv(
    "philippines_tech_jobs_raw.csv",
    index=False
)

print("\nSaved successfully.")


Batch 1 | HTTP 200
Jobs returned: 49
Batch 2 | HTTP 200
Jobs returned: 48
Batch 3 | HTTP 200
Jobs returned: 42
No more pages.

Batch 1 | HTTP 200
Jobs returned: 50
Batch 2 | HTTP 200
Jobs returned: 37
No more pages.

Batch 1 | HTTP 504
Request failed: {"message": "Endpoint request timed out"}
Batch 2 | HTTP 200
Jobs returned: 48
Batch 3 | HTTP 200
Jobs returned: 48
Batch 4 | HTTP 200
Jobs returned: 45

Batch 1 | HTTP 200
Jobs returned: 50
Batch 2 | HTTP 200
Jobs returned: 49
Batch 3 | HTTP 200
Jobs returned: 27
No more pages.

Batch 1 | HTTP 200
Jobs returned: 50
Batch 2 | HTTP 200
Jobs returned: 49
Batch 3 | HTTP 200
Jobs returned: 39
No more pages.

Raw shape:
(631, 36)

After duplicates:
(631, 36)

Saved successfully.
